In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_squared_error, mean_absolute_error, r2_score)
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")

df_mat = pd.read_excel("student-mat.xlsx")
df_por = pd.read_excel("student-por.xlsx")


# Combine both datasets
df_mat['subject'] = 'Math'
df_por['subject'] = 'Portuguese'
df = pd.concat([df_mat, df_por], ignore_index=True)

# ====================== 2. PREPROCESSING ======================
df = df.drop(['G1', 'G2'], axis=1)
X = df.drop('G3', axis=1)
y_reg = df['G3']
y_clf = (df['G3'] >= 10).astype(int)   # 1 = Pass, 0 = Fail

# Encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
le_dict = {}   # save encoders if needed later
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

print("\nFeatures after encoding:", X.shape)
print("Class distribution (Pass/Fail):\n", y_clf.value_counts())

# Train-test split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Regression Models
rf_reg = RandomForestRegressor(n_estimators=150, random_state=42, max_depth=12)
rf_reg.fit(X_train, y_reg_train)

lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)

# Classification Models
rf_clf = RandomForestClassifier(n_estimators=150, random_state=42, max_depth=12)
rf_clf.fit(X_train, y_clf_train)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_clf_train)

print("Models trained successfully!")

# ====================== 4. MAKE PREDICTIONS ======================
print("\n" + "="*70)
print("PREDICTIONS")
print("="*70) 

# ----- Regression Predictions -----
y_pred_rf_reg = rf_reg.predict(X_test)
y_pred_lin = lin_reg.predict(X_test_scaled)

print("\n--- REGRESSION PREDICTIONS (Exact G3 Grade) ---")
print(f"Random Forest  → RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_pred_rf_reg)):.4f} | R²: {r2_score(y_reg_test, y_pred_rf_reg):.4f}")
print(f"Linear Regression → RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_pred_lin)):.4f} | R²: {r2_score(y_reg_test, y_pred_lin):.4f}")

# Show sample predictions
reg_results = pd.DataFrame({
    'Actual_G3': y_reg_test.values,
    'RF_Predicted_G3': np.round(y_pred_rf_reg, 2),
    'LinReg_Predicted_G3': np.round(y_pred_lin, 2),
    'RF_Error': np.round(y_reg_test.values - y_pred_rf_reg, 2),
    'LinReg_Error': np.round(y_reg_test.values - y_pred_lin, 2)
})
print("\nSample Regression Predictions (first 15):")
print(reg_results.head(15).to_string(index=False))

# ----- Classification Predictions -----
y_pred_rf_clf = rf_clf.predict(X_test)
y_pred_log = log_reg.predict(X_test_scaled)

print("\n--- CLASSIFICATION PREDICTIONS (Pass / Fail) ---")
print(f"Random Forest Accuracy     : {accuracy_score(y_clf_test, y_pred_rf_clf):.4f}")
print(f"Logistic Regression Accuracy: {accuracy_score(y_clf_test, y_pred_log):.4f}")

# Show sample predictions
clf_results = pd.DataFrame({
    'Actual_G3': y_reg_test.values,
    'Actual_Result': y_clf_test.map({1: 'Pass', 0: 'Fail'}).values,
    'RF_Predicted': pd.Series(y_pred_rf_clf).map({1: 'Pass', 0: 'Fail'}).values,
    'LogReg_Predicted': pd.Series(y_pred_log).map({1: 'Pass', 0: 'Fail'}).values
})
print("\nSample Classification Predictions (first 15):")
print(clf_results.head(15).to_string(index=False))


# ====================== 5. FULL EVALUATION SUMMARY ======================
print("\n" + "="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)

print("\nCLASSIFICATION (Accuracy - higher is better)")
print(f"{'Model':<25} {'Accuracy':>10}")
print("-" * 37)
print(f"{'Random Forest':<25} {accuracy_score(y_clf_test, y_pred_rf_clf):>10.4f}")
print(f"{'Logistic Regression':<25} {accuracy_score(y_clf_test, y_pred_log):>10.4f}")

print("\nREGRESSION (RMSE lower better | R² higher better)")
print(f"{'Model':<25} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
print("-" * 57)
print(f"{'Random Forest':<25} {np.sqrt(mean_squared_error(y_reg_test, y_pred_rf_reg)):>10.4f} "
      f"{mean_absolute_error(y_reg_test, y_pred_rf_reg):>10.4f} {r2_score(y_reg_test, y_pred_rf_reg):>10.4f}")
print(f"{'Linear Regression':<25} {np.sqrt(mean_squared_error(y_reg_test, y_pred_lin)):>10.4f} "
      f"{mean_absolute_error(y_reg_test, y_pred_lin):>10.4f} {r2_score(y_reg_test, y_pred_lin):>10.4f}")

# Feature Importance
print("\nTop 12 Important Features (Random Forest Classifier):")
feat_imp = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(feat_imp.head(12))

Loading datasets...
Math dataset shape     : (395, 33)
Portuguese dataset shape: (649, 33)
Combined dataset shape : (1044, 34)
Missing values: 0

Features after encoding: (1044, 31)
Class distribution (Pass/Fail):
 G3
1    814
0    230
Name: count, dtype: int64

Training models...
Models trained successfully!

PREDICTIONS

--- CLASSIFICATION PREDICTIONS (Pass / Fail) ---
Random Forest Accuracy     : 0.7943
Logistic Regression Accuracy: 0.7943

Sample Classification Predictions (first 15):
 Actual_G3 Actual_Result RF_Predicted LogReg_Predicted
        10          Pass         Pass             Pass
        15          Pass         Pass             Pass
        16          Pass         Pass             Pass
         8          Fail         Pass             Pass
        10          Pass         Pass             Pass
        14          Pass         Pass             Pass
        10          Pass         Pass             Pass
        11          Pass         Pass             Pass
        14 